# Does the clean-data fusion survive the official robustness ladder? (CLIP + DINOv2 only)

The 3-expert version of this notebook got stuck for 4.5+ hours on AEROBLADE's
ladder extraction with zero log output (byteprint's extract() only logs at INFO
level, which wasn't enabled) -- genuinely unclear whether it was still silently
working or actually hung, and not worth the GPU time to find out. This version
drops AEROBLADE and answers the CLIP+DINOv2 half of the question first: does a
fusion model calibrated on clean images still make good decisions once inputs
are JPEG-compressed, blurred, resized, noised, color-jittered, or cropped
(`OFFICIAL_LADDER` from `byteprint/launder.py`, the brief's fixed §5.2 spec)?

Design: fit the fusion model once on clean-image scores only (443 images), then
apply that fixed model to every rung's degraded scores and report per-rung AUC /
TPR@1%FPR -- matching BYTEPRINT's own `--by-spec` convention (never pool rungs).

**Before running:** Settings -> Accelerator -> GPU T4 x2 (machine_shape pinned).
Attach `byteprint-realdata`, `byteprint-code`, `clip-reactivity-code`. Verbose
(`-v`) logging is on this time so a long-running step actually prints progress.

In [ ]:
import time
t0 = time.time()
def checkpoint(label):
    print(f'[+{time.time()-t0:7.1f}s] {label}', flush=True)

checkpoint('notebook start')

import torch
assert torch.cuda.is_available(), 'no GPU visible -- check Settings > Accelerator'
cap = torch.cuda.get_device_capability()
print(f'GPU: {torch.cuda.get_device_name()}  compute capability {cap}')
assert cap >= (7, 0), (
    f'compute capability {cap} is too old for this PyTorch build (needs >=7.0) -- '
    'P100-vs-T4 problem. Check machine_shape in kernel-metadata.json.'
)
checkpoint('GPU check passed')

In [ ]:
import glob, os, shutil

def find_code(marker_file, dest):
    candidates = [
        os.path.dirname(p) for p in glob.glob(f'/kaggle/input/**/{marker_file}', recursive=True)
    ]
    assert candidates, f'attach the dataset containing {marker_file} to this notebook'
    shutil.copytree(candidates[0], dest)
    return dest

byteprint_dir = find_code('pyproject.toml', '/kaggle/working/byteprint')
clip_dir = find_code('production_pipeline.py', '/kaggle/working/clip_src')
checkpoint(f'copied byteprint source -> {byteprint_dir}, clip source -> {clip_dir}')

%cd /kaggle/working/byteprint
!pip install -q -e .
assert _exit_code == 0, f'byteprint install failed, exit code {_exit_code}'
!pip install -q transformers scikit-image
assert _exit_code == 0, f'pip install failed, exit code {_exit_code}'
!pip install -q scikit-learn==1.9.0
assert _exit_code == 0, f'scikit-learn pin failed, exit code {_exit_code}'
checkpoint('pip installs done')

In [ ]:
import sys
sys.path.insert(0, clip_dir)
sys.path.insert(0, byteprint_dir)

try:
    import production_pipeline  # noqa: F401
    import byteprint.cli  # noqa: F401
    from byteprint.launder import OFFICIAL_LADDER, apply as launder_apply
except ImportError as e:
    raise RuntimeError(f'import sanity check failed: {e}') from e
print('OFFICIAL_LADDER:', OFFICIAL_LADDER)
checkpoint('import sanity check passed')

In [ ]:
train_candidates = [
    p for p in glob.glob('/kaggle/input/**/train', recursive=True)
    if os.path.isdir(os.path.join(p, 'real')) and os.path.isdir(os.path.join(p, 'fake'))
]
test_candidates = [
    p for p in glob.glob('/kaggle/input/**/test', recursive=True)
    if os.path.isdir(os.path.join(p, 'real')) and os.path.isdir(os.path.join(p, 'fake'))
]
assert train_candidates and test_candidates, 'attach the byteprint-realdata dataset first'
train_dir, test_dir = train_candidates[0], test_candidates[0]

!find {train_dir} -type f | wc -l
!find {test_dir} -type f | wc -l
checkpoint(f'found data: train={train_dir} test={test_dir}')

## 1. CLIP + reactivity-delta -- score every image under every official ladder rung

In [ ]:
import json
import numpy as np
from PIL import Image

os.chdir(clip_dir)
from production_pipeline import predict_proba

def collect_split(root):
    records = []
    real_dir = os.path.join(root, 'real')
    for f in sorted(os.listdir(real_dir)):
        records.append((os.path.join(real_dir, f), 0))
    fake_root = os.path.join(root, 'fake')
    for source in sorted(os.listdir(fake_root)):
        for f in sorted(os.listdir(os.path.join(fake_root, source))):
            records.append((os.path.join(fake_root, source, f), 1))
    return records

test_records = collect_split(test_dir)
test_paths = [r[0] for r in test_records]
test_labels = [r[1] for r in test_records]
print(f'{len(test_records)} test images x {len(OFFICIAL_LADDER)} rungs')

clip_by_rung = {}
for rung_idx, rung in enumerate(OFFICIAL_LADDER):
    rung_scores = []
    chunk = 64
    for i in range(0, len(test_paths), chunk):
        batch_paths = test_paths[i:i+chunk]
        imgs = []
        for p in batch_paths:
            arr = np.asarray(Image.open(p).convert('RGB'))
            arr = launder_apply(arr, rung, seed=0)
            imgs.append(Image.fromarray(arr))
        probs, _ = predict_proba(imgs, batch_size=32, model_dir=os.path.join(clip_dir, 'model'))
        rung_scores.extend(float(p) for p in probs)
    clip_by_rung[rung] = rung_scores
    checkpoint(f'CLIP rung {rung_idx+1}/{len(OFFICIAL_LADDER)} ({rung}) done')

clip_ladder_out = {
    'paths': test_paths, 'labels': test_labels,
    'by_rung': clip_by_rung,
}
with open('/kaggle/working/clip_ladder_scores.json', 'w') as f:
    json.dump(clip_ladder_out, f)
checkpoint('CLIP ladder scoring done -> clip_ladder_scores.json')
os.chdir('/kaggle/working/byteprint')

## 2. DINOv2 -- train on the full 2506 train images, score every test image under every rung

In [ ]:
!python -m byteprint.cli -v extract --data {train_dir} --cache /kaggle/working/cache/dino_train \
    --expert dinov2 --augment 2 --device cuda --batch-size 64 --seed 0
assert _exit_code == 0, f'dinov2 train extract failed, exit code {_exit_code}'
checkpoint('dinov2 train extract done (full 2506 images)')

!python -m byteprint.cli -v extract --data {test_dir} --cache /kaggle/working/cache/dino_test_ladder \
    --expert dinov2 --ladder official --device cuda --batch-size 64
assert _exit_code == 0, f'dinov2 test ladder extract failed, exit code {_exit_code}'
checkpoint('dinov2 test extract done (443 images x 15 rungs)')

In [ ]:
!python -m byteprint.cli train --cache /kaggle/working/cache/dino_train \
    --out /kaggle/working/runs/probe.joblib --target-fpr 0.01
assert _exit_code == 0, f'probe training failed, exit code {_exit_code}'
checkpoint('dinov2 probe trained')

In [ ]:
sys.path.insert(0, '/kaggle/working/byteprint')
from byteprint.cache import EmbeddingStore, ExtractConfig
from byteprint.probe import LinearProbe

config = ExtractConfig(**json.loads(open('/kaggle/working/cache/dino_test_ladder/config.json').read()))
store = EmbeddingStore.open('/kaggle/working/cache/dino_test_ladder', config)
probe = LinearProbe.load('/kaggle/working/runs/probe.joblib')

features = store.matrix()
scores = probe.score(features)
dino_ladder_out = {
    'paths': store.paths(), 'labels': [int(l) for l in store.labels()],
    'specs': store.specs(), 'scores': [float(s) for s in scores],
}
with open('/kaggle/working/dino_ladder_scores.json', 'w') as f:
    json.dump(dino_ladder_out, f)
checkpoint(f'DINOv2 ladder scoring done, {len(scores)} rows -> dino_ladder_scores.json')

## 3. Fit fusion on clean scores only, then apply it -- unmodified -- to every rung

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

clip_ladder = json.load(open('/kaggle/working/clip_ladder_scores.json'))
dino_ladder = json.load(open('/kaggle/working/dino_ladder_scores.json'))

clip_map = {rung: dict(zip(clip_ladder['paths'], scores)) for rung, scores in clip_ladder['by_rung'].items()}
dino_map = {}
for p, spec, s in zip(dino_ladder['paths'], dino_ladder['specs'], dino_ladder['scores']):
    dino_map.setdefault(spec, {})[p] = s
label_map = dict(zip(clip_ladder['paths'], clip_ladder['labels']))

def tpr_at_fpr(y_true, scores, target_fpr=0.01):
    fpr, tpr, _ = roc_curve(y_true, scores)
    idx = max(np.searchsorted(fpr, target_fpr, side='right') - 1, 0)
    return tpr[idx]

clean_paths = sorted(clip_map['none'])
y_clean = np.array([label_map[p] for p in clean_paths])
X_clean = np.column_stack([
    [clip_map['none'][p] for p in clean_paths],
    [dino_map['none'][p] for p in clean_paths],
])
final_scaler = StandardScaler().fit(X_clean)
final_fusion = LogisticRegression(max_iter=2000).fit(final_scaler.transform(X_clean), y_clean)
checkpoint('fusion model fit on clean rung, ready to apply to every rung')

In [ ]:
print(f'{"rung":<12}{"CLIP":>10}{"DINOv2":>10}{"FUSED":>10}   (AUC)')
rows = {}
for rung in OFFICIAL_LADDER:
    paths = sorted(set(clip_map[rung]) & set(dino_map[rung]))
    y = np.array([label_map[p] for p in paths])
    x_clip = np.array([clip_map[rung][p] for p in paths])
    x_dino = np.array([dino_map[rung][p] for p in paths])
    X = np.column_stack([x_clip, x_dino])
    fused = final_fusion.predict_proba(final_scaler.transform(X))[:, 1]

    row = {
        'n': len(paths),
        'clip_auc': roc_auc_score(y, x_clip), 'clip_tpr': tpr_at_fpr(y, x_clip),
        'dino_auc': roc_auc_score(y, x_dino), 'dino_tpr': tpr_at_fpr(y, x_dino),
        'fused_auc': roc_auc_score(y, fused), 'fused_tpr': tpr_at_fpr(y, fused),
    }
    rows[rung] = row
    print(f'{rung:<12}{row["clip_auc"]:>10.4f}{row["dino_auc"]:>10.4f}{row["fused_auc"]:>10.4f}   (n={row["n"]})')

print()
print(f'{"rung":<12}{"CLIP":>10}{"DINOv2":>10}{"FUSED":>10}   (TPR@1%FPR)')
for rung in OFFICIAL_LADDER:
    row = rows[rung]
    print(f'{rung:<12}{row["clip_tpr"]:>10.4f}{row["dino_tpr"]:>10.4f}{row["fused_tpr"]:>10.4f}')

mean_row = {k: float(np.mean([rows[r][k] for r in OFFICIAL_LADDER])) for k in
            ['clip_auc','dino_auc','fused_auc','clip_tpr','dino_tpr','fused_tpr']}
print()
print('mean across all 15 rungs:', json.dumps(mean_row, indent=2))
checkpoint('per-rung ladder evaluation done')

In [ ]:
with open('/kaggle/working/ladder_results_by_rung_2expert.json', 'w') as f:
    json.dump({'per_rung': rows, 'mean_across_rungs': mean_row}, f, indent=2)

!tar -czf /kaggle/working/fusion_ladder_2expert_results.tar.gz -C /kaggle/working \
    clip_ladder_scores.json dino_ladder_scores.json \
    ladder_results_by_rung_2expert.json runs
print('done -- download fusion_ladder_2expert_results.tar.gz from the notebook output panel')
checkpoint('results packaged')